# Routing using semantic similarity

In [2]:
from langchain_community.utils.math import cosine_similarity
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import OpenAIEmbeddings
from langchain_groq import ChatGroq

physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{query}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{query}"""

embeddings = OpenAIEmbeddings()
prompt_templates = [physics_template, math_template]
prompt_embeddings = embeddings.embed_documents(prompt_templates)


def prompt_router(input):
    query_embedding = embeddings.embed_query(input["query"])
    similarity = cosine_similarity([query_embedding], prompt_embeddings)[0]
    most_similar = prompt_templates[similarity.argmax()]
    print("Using MATH" if most_similar == math_template else "Using PHYSICS")
    return PromptTemplate.from_template(most_similar)


chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | ChatGroq(model_name='llama3-70b-8192')
    | StrOutputParser()
)

In [3]:
print(chain.invoke("What's a black hole"))

Using PHYSICS
Black holes are one of the most fascinating and mind-bending concepts in all of physics.

A black hole is a region in space where the gravitational pull is so strong that nothing, including light, can escape. It's formed when a massive star collapses in on itself and its gravity becomes so strong that it warps the fabric of spacetime around it.

Imagine spacetime as a trampoline. When you place a heavy object, like a star, on the trampoline, it warps the surface, creating a dent. The more massive the star, the deeper the dent. If the star is massive enough, the dent becomes so deep that it creates a hole in the trampoline, and that's essentially what a black hole is.

The point of no return, where the gravity is so strong that escape is impossible, is called the event horizon. Once you cross the event horizon, you're trapped, and you'll be pulled towards the center of the black hole, known as the singularity.

The singularity is a point of infinite density and zero volume

In [4]:
print(chain.invoke("What's a path integral"))

Using MATH
What a great question! A path integral is a fundamental concept in mathematics and physics, and it's a pleasure to break it down and explain it in detail.

**Breaking it down:**

To understand a path integral, we need to tackle three main components:

1. **Integrals**: A path integral is a type of integral, so let's start with a brief review of integrals. An integral is a mathematical operation that computes the area under a curve or the accumulation of a quantity over an interval. There are different types of integrals, such as definite integrals, indefinite integrals, and line integrals.
2. **Paths**: In the context of path integrals, a path refers to a curve or a trajectory in a mathematical space, such as a function space or a configuration space. Think of a path as a sequence of points that connect to form a continuous curve.
3. **Quantum Mechanics**: Path integrals are deeply rooted in quantum mechanics, which is a fundamental theory in physics. In quantum mechanics, p